# How to collect Run 3 data?

In [1]:
import numpy as np
from trig_egamma_frame import ElectronLoop, DataframeSchemma, EventContext, GeV
from trig_egamma_frame.algorithms.dumper import ElectronDumper_v2 as ElectronDumper
from trig_egamma_frame import Filter
from trig_egamma_frame.kernel import ToolSvc

Welcome to JupyROOT 6.30/04


I0000 00:00:1776207825.343152 3617131 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776207825.345259 3617131 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776207825.420747 3617131 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776207830.181557 3617131 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
class MyFilter:
    def __init__ ( self, 
                  is_data : bool=False, 
                  is_jets : bool=False
                  ):
        self.is_data = is_data
        self.is_jets = is_jets

    def __call__(self, ctx : EventContext) -> bool:

        elCont = ctx.getHandler( "ElectronContainer" )
        if elCont.et() < 2*GeV:
            return False
        fc = ctx.getHandler( "HLT__TrigEMClusterContainer" )
        if not fc.isGoodRinger():
            return False

        if self.is_data:
            return True
        
        mc = ctx.getHandler("MonteCarloContainer")
        if self.is_jets:
            if mc.isTruthElectronFromAny():
                return False
        else:
            if fc.et() < 15*GeV: # Jpsiee
                if not mc.isTruthElectronFromJpsiPrompt():
                    return False
            else: # Zee
                if not mc.isTruthElectronFromZ():
                    return False

        return True

In [3]:
def isZ_decorator( ctx : EventContext ) -> np.int32:
    mc = ctx.getHandler("MonteCarloContainer")
    return np.int32(mc.isTruthElectronFromZ())
def isAny_decorator( ctx : EventContext ) -> np.int32:
    mc = ctx.getHandler("MonteCarloContainer")
    return np.int32(mc.isTruthElectronFromAny())
def isJpsi_decorator( ctx : EventContext ) -> np.int32:
    mc = ctx.getHandler("MonteCarloContainer")
    return np.int32(mc.isTruthElectronFromJpsiPrompt())
def target( ctx : EventContext ) -> np.int32:
    return np.int32(0) if is_jets else np.int32(1)

In [4]:
input_files = "/mnt/shared/storage03/projects/cern/data/mc21_13p6TeV/no_restrictions/user.isilvafe.mc21_13p6TeV.Zee.AOD.r14136_2sigma_constraint_26.10.23_v0_EXT0.tap_zee_5M_XYZ.root.tgz/*.root"
output = 'output'
et_bins  = [3., 7., 10., 15., 20., 30., 40., 50., 1000000.]
eta_bins = [0.0, 0.8, 1.37, 1.54, 2.37, 2.50]
triggers = [
   "HLT_e28_lhtight_nod0_ringer_v1_ivarloose_eEM24VHI"
]

In [5]:
acc = ElectronLoop( "EventATLASLoop",
                    inputFile  = input_files,
                    treePath  =  "*/HLT/EgammaMon/summary/events",
                    dataframe  = DataframeSchemma.Run3,
                    outputFile = "output.root",
                    abort = True,
                  )

my_filter = MyFilter(is_data=True, is_jets=False)

ToolSvc+=Filter( "Filter", [my_filter])

dumper = ElectronDumper(output, et_bins, eta_bins)

for trigName in triggers:
    dumper.decorate_chain(trigName)

dumper.decorate( "mc_isTruthElectronFromZ"          , isZ_decorator   )
dumper.decorate( "mc_isTruthElectronFromAny"        , isAny_decorator )
dumper.decorate( "mc_isTruthElectronFromJpsiPromt"  , isJpsi_decorator)
dumper.decorate( "target", target)

ToolSvc+=dumper

acc.run(1000)

{'FSinfo': '',
 'IDinfo': 'lhtight',
 'IDinfoType': '',
 'L1Legacy': False,
 'L1Threshold': 'eEM24VHI',
 'L1threshold': '',
 'L2IDAlg': '',
 'addInfo': [],
 'alignmentGroup': ['Electron'],
 'caloInfo': '',
 'etaRange': '0eta250',
 'extra': '',
 'gsfInfo': '',
 'hypoInfo': '',
 'idperfInfo': '',
 'isoInfo': 'ivarloose',
 'lhInfo': 'nod0',
 'lrtInfo': '',
 'multiplicity': '',
 'reccalibInfo': '',
 'recoAlg': '',
 'ringerVersion': 'run3_v1',
 'sigFolder': ['Egamma'],
 'signature': 'Electron',
 'subSigs': ['Electron'],
 'threshold': 28.0,
 'tnpInfo': '',
 'topo': [],
 'trigName': 'HLT_e28_lhtight_nod0_ringer_v1_ivarloose_eEM24VHI',
 'trigType': 'e'}
14-Apr-2026 20:06:31 | Signature :e
14-Apr-2026 20:06:31 | Threshold :28.0
14-Apr-2026 20:06:31 | Pidname   :lhtight
14-Apr-2026 20:06:31 | noringerinfo :
14-Apr-2026 20:06:31 | Configure ringer
14-Apr-2026 20:06:31 | Threshold :28.0
14-Apr-2026 20:06:31 | Pidname   :lhtight
14-Apr-2026 20:06:31 | Configure nominal...
14-Apr-2026 20:06:31 | Ele

creating collection tree : 100%|███████████████████████████████████████████████████████████████████| 496/496 [00:02<00:00, 205.70it/s]


14-Apr-2026 20:06:35 | creating StoreGate...
14-Apr-2026 20:06:35 | Creating containers...
14-Apr-2026 20:06:35 | Initialize the dataframe with name: EventInfoContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: MonteCarloContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: CaloClusterContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: MenuContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: HLT__TrigEMClusterContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: HLT__CaloClusterContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: HLT__EmTauRoIContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: ElectronContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: TrackParticleContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: HLT__TrigElectronContainer
14-Apr-2026 20:06:35 | Initialize the dataframe with name: HLT__ElectronContainer
14-Apr-2026 20:06:

ValueError: could not convert string to float: ''